In [1]:
import os
import random
import warnings
import numpy as np
import pandas as pd

In [2]:
# Loading the dataset
df = pd.read_csv('archive/games.csv')

# Reading top few rows
df.head()

,AppID,Name,Release date,Estimated owners,Peak CCU,Required age,Price,DiscountDLC count,About the game,Supported languages,...,Average playtime two weeks,Median playtime forever,Median playtime two weeks,Developers,Publishers,Categories,Genres,Tags,Screenshots,Movies
2539430,Black Dragon Mage Playtest,"Aug 1, 2023",0 - 0,0,0,0.00,0,0,NaN,[],...,0,0,0,NaN,NaN,NaN,NaN,NaN,https://shared.akamai.steamstatic.com/store_it...,NaN
496350,Supipara - Chapter 1 Spring Has Come!,"Jul 29, 2016",0 - 20000,0,0,5.24,65,0,"Springtime, April: when the cherry trees come ...",['English'],...,0,8,0,minori,MangaGamer,"Single-player,Steam Trading Cards,Steam Cloud,...",Adventure,"Adventure,Visual Novel,Anime,Cute",https://shared.akamai.steamstatic.com/store_it...,NaN
1034400,Mystery Solitaire The Black Raven,"May 6, 2019",0 - 20000,0,0,4.99,0,0,"Immerse yourself in the most beloved, mystical...","['English', 'French', 'German', 'Russian']",...,0,0,0,Somer Games,8floor,"Single-player,Family Sharing",Casual,"Casual,Card Game,Solitaire,Puzzle,Hidden Objec...",https://shared.akamai.steamstatic.com/store_it...,NaN
3292190,버튜버 파라노이아 - Vtuber Paranoia,"Oct 31, 2024",0 - 20000,1,0,8.99,0,1,"synopsis 'Hello, I'm Hiyoro, a new YouTuber!' ...",['Korean'],...,0,0,0,유진게임즈,유진게임즈,"Single-player,Steam Achievements,Family Sharing","Casual,Indie,Simulation",NaN,https://shared.akamai.steamstatic.com/store_it...,NaN
3631080,Maze Quest VR,"Apr 24, 2025",0 - 20000,0,0,4.99,0,0,Its not just a Maze; its a Quest! Enter the ca...,['English'],...,0,0,0,Reality Expanded LLC,Reality Expanded LLC,"Single-player,VR Only,Steam Leaderboards,Famil...","Action,Early Access",NaN,https://shared.akamai.steamstatic.com/store_it...,NaN


In [4]:
df.columns.tolist()

['AppID',
 'Name',
 'Release date',
 'Estimated owners',
 'Peak CCU',
 'Required age',
 'Price',
 'DiscountDLC count',
 'About the game',
 'Supported languages',
 'Full audio languages',
 'Reviews',
 'Header image',
 'Website',
 'Support url',
 'Support email',
 'Windows',
 'Mac',
 'Linux',
 'Metacritic score',
 'Metacritic url',
 'User score',
 'Positive',
 'Negative',
 'Score rank',
 'Achievements',
 'Recommendations',
 'Notes',
 'Average playtime forever',
 'Average playtime two weeks',
 'Median playtime forever',
 'Median playtime two weeks',
 'Developers',
 'Publishers',
 'Categories',
 'Genres',
 'Tags',
 'Screenshots',
 'Movies']

In [6]:
# Remove columns not needed for ER/relational modeling
cols_to_remove = [
    'Reviews',
    'Header image',
    'Recommendations',
    'Screenshots',
    'Movies',
    'Score rank',
    'Metacritic url',
    'Notes',
    'Website',
    'Support url',
    'Support email',
]

existing = [c for c in cols_to_remove if c in df.columns]
missing = [c for c in cols_to_remove if c not in df.columns]

df_clean = df.drop(columns=existing).copy()

print('Removed columns:', existing)
if missing:
    print('Not found in dataset:', missing)

print('Original shape:', df.shape)
print('Cleaned shape :', df_clean.shape)

# Save a cleaned version for Stage 2 modeling
output_path = 'archive/games_clean_stage2.csv'
df_clean.to_csv(output_path, index=False)
print('Saved cleaned file to:', output_path)

# Quick check
df_clean.head()

Removed columns: ['Reviews', 'Header image', 'Recommendations', 'Screenshots', 'Movies', 'Score rank', 'Metacritic url', 'Notes', 'Website', 'Support url', 'Support email']
Original shape: (125855, 39)
Cleaned shape : (125855, 28)
Saved cleaned file to: archive/games_clean_stage2.csv


,AppID,Name,Release date,Estimated owners,Peak CCU,Required age,Price,DiscountDLC count,About the game,Supported languages,...,Achievements,Average playtime forever,Average playtime two weeks,Median playtime forever,Median playtime two weeks,Developers,Publishers,Categories,Genres,Tags
2539430,Black Dragon Mage Playtest,"Aug 1, 2023",0 - 0,0,0,0.00,0,0,NaN,[],...,0,0,0,0,0,NaN,NaN,NaN,NaN,NaN
496350,Supipara - Chapter 1 Spring Has Come!,"Jul 29, 2016",0 - 20000,0,0,5.24,65,0,"Springtime, April: when the cherry trees come ...",['English'],...,0,8,0,8,0,minori,MangaGamer,"Single-player,Steam Trading Cards,Steam Cloud,...",Adventure,"Adventure,Visual Novel,Anime,Cute"
1034400,Mystery Solitaire The Black Raven,"May 6, 2019",0 - 20000,0,0,4.99,0,0,"Immerse yourself in the most beloved, mystical...","['English', 'French', 'German', 'Russian']",...,0,0,0,0,0,Somer Games,8floor,"Single-player,Family Sharing",Casual,"Casual,Card Game,Solitaire,Puzzle,Hidden Objec..."
3292190,버튜버 파라노이아 - Vtuber Paranoia,"Oct 31, 2024",0 - 20000,1,0,8.99,0,1,"synopsis 'Hello, I'm Hiyoro, a new YouTuber!' ...",['Korean'],...,19,0,0,0,0,유진게임즈,유진게임즈,"Single-player,Steam Achievements,Family Sharing","Casual,Indie,Simulation",NaN
3631080,Maze Quest VR,"Apr 24, 2025",0 - 20000,0,0,4.99,0,0,Its not just a Maze; its a Quest! Enter the ca...,['English'],...,0,0,0,0,0,Reality Expanded LLC,Reality Expanded LLC,"Single-player,VR Only,Steam Leaderboards,Famil...","Action,Early Access",NaN


In [7]:
# Profile cleaned columns to decide further removals
null_pct = (df_clean.isna().mean() * 100).sort_values(ascending=False)
print('Columns in cleaned data:', len(df_clean.columns))
print(df_clean.columns.tolist())
print('\nTop 12 columns by missing %:')
print(null_pct.head(12).round(2))

Columns in cleaned data: 28
['AppID', 'Name', 'Release date', 'Estimated owners', 'Peak CCU', 'Required age', 'Price', 'DiscountDLC count', 'About the game', 'Supported languages', 'Full audio languages', 'Windows', 'Mac', 'Linux', 'Metacritic score', 'User score', 'Positive', 'Negative', 'Achievements', 'Average playtime forever', 'Average playtime two weeks', 'Median playtime forever', 'Median playtime two weeks', 'Developers', 'Publishers', 'Categories', 'Genres', 'Tags']

Top 12 columns by missing %:
Tags                33.77
Categories           7.12
Publishers           7.09
About the game       6.72
Developers           6.71
Genres               6.69
AppID                0.00
Price                0.00
Estimated owners     0.00
Peak CCU             0.00
Release date         0.00
Name                 0.00
dtype: float64


## Stage 2 - E/R Model (Steam Games Dataset)

### 1) Core Entities and Attributes

**Game**
- **PK:** app_id
- name
- release_date
- estimated_owners
- peak_ccu
- required_age
- price
- metacritic_score
- user_score
- positive
- negative
- achievements
- avg_playtime_forever
- avg_playtime_2weeks
- median_playtime_forever
- median_playtime_2weeks
- windows_support
- mac_support
- linux_support
- about_the_game
- discount_percent *(optional; only if split from malformed source column)*
- dlc_count *(optional; only if split from malformed source column)*

**Developer**
- **PK:** developer_id
- developer_name

**Publisher**
- **PK:** publisher_id
- publisher_name

**Genre**
- **PK:** genre_id
- genre_name

**Category**
- **PK:** category_id
- category_name

**Tag**
- **PK:** tag_id
- tag_name

**Language**
- **PK:** language_id
- language_name

**AudioLanguage**
- **PK:** audio_language_id
- audio_language_name

### 2) Relationships and Cardinalities

- One **Game** can have many **Developers**, and one **Developer** can develop many games (M:N).
- One **Game** can have many **Publishers**, and one **Publisher** can publish many games (M:N).
- One **Game** can have many **Genres**, and one **Genre** can belong to many games (M:N).
- One **Game** can have many **Categories**, and one **Category** can belong to many games (M:N).
- One **Game** can have many **Tags**, and one **Tag** can belong to many games (M:N).
- One **Game** can support many **Languages**, and one **Language** can be supported by many games (M:N).
- One **Game** can support many **AudioLanguages**, and one **AudioLanguage** can be supported by many games (M:N).

### 3) Associative Entities (for M:N relationships)

- **GameDeveloper**(app_id, developer_id)
- **GamePublisher**(app_id, publisher_id)
- **GameGenre**(app_id, genre_id)
- **GameCategory**(app_id, category_id)
- **GameTag**(app_id, tag_id)
- **GameLanguage**(app_id, language_id)
- **GameAudioLanguage**(app_id, audio_language_id)

Each associative table uses a composite primary key from both foreign keys.

### 4) E/R Diagram (Chen-Style Symbols)

This follows your module style: **rectangles = entities**, **ellipses = attributes**, **diamonds = relationships**, **lines = connections**.

```mermaid
flowchart LR
    %% Entities (rectangles)
    GAME[Game]
    DEV[Developer]
    PUB[Publisher]
    GEN[Genre]
    CAT[Category]
    TAG[Tag]
    LANG[Language]
    ALANG[AudioLanguage]

    %% Game attributes (ellipses)
    A1((app_id PK))
    A2((name))
    A3((release_date))
    A4((price))
    A5((estimated_owners))
    A6((metacritic_score))
    A7((user_score))
    A8((positive))
    A9((negative))
    A10((achievements))
    A11((windows_support))
    A12((mac_support))
    A13((linux_support))

    %% Other entity attributes (ellipses)
    D1((developer_id PK))
    D2((developer_name))
    P1((publisher_id PK))
    P2((publisher_name))
    G1((genre_id PK))
    G2((genre_name))
    C1((category_id PK))
    C2((category_name))
    T1((tag_id PK))
    T2((tag_name))
    L1((language_id PK))
    L2((language_name))
    AL1((audio_language_id PK))
    AL2((audio_language_name))

    %% Relationships (diamonds)
    R1{develops}
    R2{publishes}
    R3{classified_as}
    R4{includes_category}
    R5{tagged_with}
    R6{supports_language}
    R7{supports_audio_language}

    %% Attribute connections
    GAME --- A1
    GAME --- A2
    GAME --- A3
    GAME --- A4
    GAME --- A5
    GAME --- A6
    GAME --- A7
    GAME --- A8
    GAME --- A9
    GAME --- A10
    GAME --- A11
    GAME --- A12
    GAME --- A13

    DEV --- D1
    DEV --- D2
    PUB --- P1
    PUB --- P2
    GEN --- G1
    GEN --- G2
    CAT --- C1
    CAT --- C2
    TAG --- T1
    TAG --- T2
    LANG --- L1
    LANG --- L2
    ALANG --- AL1
    ALANG --- AL2

    %% Entity-relationship connections
    GAME --- R1 --- DEV
    GAME --- R2 --- PUB
    GAME --- R3 --- GEN
    GAME --- R4 --- CAT
    GAME --- R5 --- TAG
    GAME --- R6 --- LANG
    GAME --- R7 --- ALANG
```

### 5) Modeling Notes for Report

- `Tags` is intentionally kept because it captures community labeling and is useful for query features in the web app.
- `Developers`, `Publishers`, `Genres`, `Categories`, `Supported languages`, and `Full audio languages` must be split into separate rows and linked through associative tables to satisfy 1NF.
- The source column `DiscountDLC count` appears malformed in the CSV header and should be cleaned into two attributes (`discount_percent`, `dlc_count`) before final SQL import, or documented as an unresolved source issue if excluded.

## Stage 2 - Submission-Friendly E/R Diagram

Use this compact version in your report screenshot. It keeps Chen-style symbols while reducing clutter.

```mermaid
flowchart LR
    %% Entities (rectangles)
    GAME[Game]
    DEV[Developer]
    PUB[Publisher]
    GEN[Genre]
    CAT[Category]
    TAG[Tag]
    LANG[Language]
    ALANG[AudioLanguage]

    %% Key attributes only (ellipses)
    GPK((app_id PK))
    GNM((name))
    GREL((release_date))
    GPR((price))
    GUS((user_score))
    GPOS((positive))
    GNEG((negative))

    DPK((developer_id PK))
    DNM((developer_name))
    PPK((publisher_id PK))
    PNM((publisher_name))
    GNPK((genre_id PK))
    GNNM((genre_name))
    CPK((category_id PK))
    CNM((category_name))
    TPK((tag_id PK))
    TNM((tag_name))
    LPK((language_id PK))
    LNM((language_name))
    APK((audio_language_id PK))
    ANM((audio_language_name))

    %% Relationships (diamonds)
    R1{develops}
    R2{publishes}
    R3{classified_as}
    R4{includes_category}
    R5{tagged_with}
    R6{supports_language}
    R7{supports_audio_language}

    %% Connections: entity to attributes
    GAME --- GPK
    GAME --- GNM
    GAME --- GREL
    GAME --- GPR
    GAME --- GUS
    GAME --- GPOS
    GAME --- GNEG

    DEV --- DPK
    DEV --- DNM
    PUB --- PPK
    PUB --- PNM
    GEN --- GNPK
    GEN --- GNNM
    CAT --- CPK
    CAT --- CNM
    TAG --- TPK
    TAG --- TNM
    LANG --- LPK
    LANG --- LNM
    ALANG --- APK
    ALANG --- ANM

    %% Connections: entities to relationships
    GAME --- R1 --- DEV
    GAME --- R2 --- PUB
    GAME --- R3 --- GEN
    GAME --- R4 --- CAT
    GAME --- R5 --- TAG
    GAME --- R6 --- LANG
    GAME --- R7 --- ALANG
```

### Cardinality Notes (for marker clarity)
- Game to Developer: M:N
- Game to Publisher: M:N
- Game to Genre: M:N
- Game to Category: M:N
- Game to Tag: M:N
- Game to Language: M:N
- Game to AudioLanguage: M:N

### Mapping Hint
Implement each M:N relationship with an associative table, e.g., `GameTag(app_id, tag_id)` and `GameGenre(app_id, genre_id)`.

## Stage 2 - Database Tables, Fields, and Normalisation (up to 3NF)

### 1) Final Relational Tables and Fields

**GAME**
- **PK:** `app_id`
- `name`
- `release_date`
- `estimated_owners`
- `peak_ccu`
- `required_age`
- `price`
- `metacritic_score`
- `user_score`
- `positive`
- `negative`
- `achievements`
- `average_playtime_forever`
- `average_playtime_2weeks`
- `median_playtime_forever`
- `median_playtime_2weeks`
- `windows`
- `mac`
- `linux`
- `about_the_game`
- `discount_percent` *(optional, if recovered from malformed source column)*
- `dlc_count` *(optional, if recovered from malformed source column)*

**DEVELOPER**
- **PK:** `developer_id`
- `developer_name` *(UNIQUE)*

**PUBLISHER**
- **PK:** `publisher_id`
- `publisher_name` *(UNIQUE)*

**GENRE**
- **PK:** `genre_id`
- `genre_name` *(UNIQUE)*

**CATEGORY**
- **PK:** `category_id`
- `category_name` *(UNIQUE)*

**TAG**
- **PK:** `tag_id`
- `tag_name` *(UNIQUE)*

**LANGUAGE**
- **PK:** `language_id`
- `language_name` *(UNIQUE)*

**AUDIO_LANGUAGE**
- **PK:** `audio_language_id`
- `audio_language_name` *(UNIQUE)*

**GAME_DEVELOPER**
- **PK:** (`app_id`, `developer_id`)
- **FK:** `app_id` -> GAME.`app_id`
- **FK:** `developer_id` -> DEVELOPER.`developer_id`

**GAME_PUBLISHER**
- **PK:** (`app_id`, `publisher_id`)
- **FK:** `app_id` -> GAME.`app_id`
- **FK:** `publisher_id` -> PUBLISHER.`publisher_id`

**GAME_GENRE**
- **PK:** (`app_id`, `genre_id`)
- **FK:** `app_id` -> GAME.`app_id`
- **FK:** `genre_id` -> GENRE.`genre_id`

**GAME_CATEGORY**
- **PK:** (`app_id`, `category_id`)
- **FK:** `app_id` -> GAME.`app_id`
- **FK:** `category_id` -> CATEGORY.`category_id`

**GAME_TAG**
- **PK:** (`app_id`, `tag_id`)
- **FK:** `app_id` -> GAME.`app_id`
- **FK:** `tag_id` -> TAG.`tag_id`

**GAME_LANGUAGE**
- **PK:** (`app_id`, `language_id`)
- **FK:** `app_id` -> GAME.`app_id`
- **FK:** `language_id` -> LANGUAGE.`language_id`

**GAME_AUDIO_LANGUAGE**
- **PK:** (`app_id`, `audio_language_id`)
- **FK:** `app_id` -> GAME.`app_id`
- **FK:** `audio_language_id` -> AUDIO_LANGUAGE.`audio_language_id`

### 2) Normal Form Evaluation

**1NF**
- Achieved by removing repeating groups and multi-valued fields from GAME.
- In the source data, developers/publishers/genres/categories/tags/languages were list-like values in one column.
- These were decomposed into separate entity tables plus associative tables, so each field now stores atomic values only.

**2NF**
- Achieved in all associative tables.
- For composite keys such as (`app_id`, `tag_id`), there are no non-key attributes depending on only part of the key.
- Core descriptive attributes are stored in their own entity tables, not duplicated in join tables.

**3NF**
- Achieved because non-key attributes depend only on the key, the whole key, and nothing but the key.
- Example: in GAME, attributes like `price`, `user_score`, and `peak_ccu` depend on `app_id` only.
- Names such as `genre_name` and `publisher_name` are isolated in lookup/entity tables, avoiding transitive dependencies inside GAME.

### 3) Higher Normal Forms (BCNF / 4NF)

**BCNF assessment**
- The design is effectively in BCNF for the proposed schema.
- Determinants are candidate keys in entity tables (`*_id`) and composite keys in associative tables.
- Natural-name columns are constrained as UNIQUE to avoid alternate determinant anomalies (for example `tag_name` determining `tag_id`).

**4NF assessment**
- The main multi-valued dependencies from the source (e.g., one game with many tags and many genres) are separated into independent associative tables.
- This decomposition addresses the practical 4NF concern for independent multivalued facts.
- Further decomposition is not necessary for this project because it would increase complexity without improving integrity meaningfully for the required queries.

### 4) Justification for Stopping at This Level

- The schema satisfies the assignment requirement of at least 3NF.
- It is already strong for update integrity and avoids major insertion/update/deletion anomalies.
- Additional normalization beyond this point would add implementation overhead in MySQL and the Node web app with limited analytical benefit for the project scope.